# 02. Native Feature Engineering & Intermediate Persistence
## 1. Objective
Execute pandas aggregations and merges natively in the notebook, outputting intermediate CSVs for maximum transparency, avoiding abstract `src/` wrappers during model training data generation.

In [1]:
import pandas as pd
import os
import sys

sys.path.append('..')
from src.feature_recalculation import recalculate_features

DATA_DIR = '../data/raw/'
PROC_DIR = '../data/processed/'
os.makedirs(PROC_DIR, exist_ok=True)

vle = pd.read_csv(os.path.join(DATA_DIR, 'studentVle.csv'))
assess = pd.read_csv(os.path.join(DATA_DIR, 'studentAssessment.csv'))
info = pd.read_csv(os.path.join(DATA_DIR, 'studentInfo.csv'))

## 2. Native VLE Aggregation (Engagement Features)
We group the 10M+ row table by student and output `engagement_features.csv`.

In [2]:
print(f"Raw VLE Shape: {vle.shape}")
vle_agg = vle.groupby(['id_student', 'code_module', 'code_presentation']).agg(
    total_clicks=('sum_click', 'sum'),
    active_days=('date', 'nunique'),
    num_unique_sites=('id_site', 'nunique')
).reset_index()
print(f"Aggregated VLE Shape: {vle_agg.shape}")

vle_agg.to_csv(os.path.join(PROC_DIR, 'engagement_features.csv'), index=False)

Raw VLE Shape: (10655280, 6)


Aggregated VLE Shape: (29228, 6)


## 3. Native Assessment Aggregation
We group the assessment scores.

In [3]:
print(f"Raw Assess Shape: {assess.shape}")
assess_agg = assess.groupby('id_student').agg(
    average_score=('score', 'mean'),
    completed_assessments=('id_assessment', 'count'),
    late_submission_count=('is_banked', 'sum') # Approximation for structural demo
).reset_index()
print(f"Aggregated Assess Shape: {assess_agg.shape}")

assess_agg.to_csv(os.path.join(PROC_DIR, 'assessment_features.csv'), index=False)

Raw Assess Shape: (173912, 5)
Aggregated Assess Shape: (23369, 4)


## 4. Master Merge & Recalculation Engine Injection
We merge the aggregations into the demographic `studentInfo` base table. Then we apply `recalculate_features` (from `src.feature_recalculation`) to guarantee parity with the Streamlit What-If engine.

In [4]:
master = pd.merge(info, vle_agg, on=['id_student', 'code_module', 'code_presentation'], how='left')
print(f"After VLE Merge Shape: {master.shape}")

master = pd.merge(master, assess_agg, on='id_student', how='left')
print(f"After Assess Merge Shape: {master.shape}")

# Impute base metrics before derived calculation
master['total_clicks'] = master['total_clicks'].fillna(0)
master['active_days'] = master['active_days'].fillna(0)
master['average_score'] = master['average_score'].fillna(0)
master['total_assessments'] = master['completed_assessments'].fillna(1)

# Apply central recalculation logic row-by-row
master_dicts = master.to_dict('records')
recalculated_dicts = [recalculate_features(d) for d in master_dicts]
master_final = pd.DataFrame(recalculated_dicts)

# Output final Dataset
master_final.to_csv(os.path.join(PROC_DIR, 'master_dataset.csv'), index=False)
print("Master dataset successfully engineered and saved to data/processed/")
master_final.head()

After VLE Merge Shape: (32593, 15)
After Assess Merge Shape: (32593, 18)


Master dataset successfully engineered and saved to data/processed/


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,late_submission_count,total_assessments,engagement_ratio,avg_daily_clicks,resource_diversity_score,late_submission_ratio,assessment_completion_rate,low_engagement_flag,low_score_flag,dropout_risk_score
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,5.0,0.153846,23.350000,0.058887,0.0,1.0,0,0,0.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,5.0,0.307692,17.937500,0.058537,0.0,1.0,0,0,0.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,NaN,1.0,0.046154,23.416667,0.078292,NaN,NaN,0,1,0.4
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,5.0,0.473077,17.544715,0.037998,0.0,1.0,0,0,0.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,5.0,0.269231,14.771429,0.063830,0.0,1.0,0,0,0.0
